# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Print dataset title and description
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get available record sets from the dataset
record_sets_info = dataset.record_sets
print("Available record sets and their @id:")
for rs in record_sets_info:
    print(f"- {rs['@id']}: {rs.get('name','(no name)')}")

# Review fields for each record set
for rs in record_sets_info:
    print(f"\nRecordSet @id: {rs['@id']}")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  Field @id: {field['@id']}  | Name: {field.get('name','')}  | DataType: {field.get('dataType','')}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only load if not empty
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nLoaded DataFrame for RecordSet @id: {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        print(dataframes[record_set_id].head())
    else:
        print(f"\nNo records found for RecordSet @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, and grouping data by key attributes.

In [ ]:
# Select the main record set for analysis: We use the first available record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Main DataFrame from RecordSet @id: {main_record_set_id}")
    print(df.head())

    # Select a numeric field for analysis (e.g., Age)
    # Find numeric fields from EDA overview (e.g., dataType: Integer or Float)
    numeric_field_id = None
    group_field_id = None
    for rs in dataset.record_sets:
        if rs['@id'] == main_record_set_id:
            for field in rs.get('fields', []):
                if field.get('dataType') in ['schema:Integer', 'schema:Float']:
                    if field['@id'] in df.columns:
                        numeric_field_id = field['@id']
                        print(f"Selected numeric field @id: {numeric_field_id}")
                        break
            # Also look for a categorical field (e.g., Sex)
            for field in rs.get('fields', []):
                if field.get('dataType') == 'schema:Text':
                    if field['@id'] in df.columns:
                        group_field_id = field['@id']
                        break
            break

    # EDA example: Filter records (e.g., Age > 50)
    threshold = 50
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped average {numeric_field_id} by {group_field_id}:")
            print(grouped_df)
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[main_record_set_id]
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    if numeric_field_id and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded FAIR^2 clinicopathological dataset using `mlcroissant` and accessed metadata.
- Reviewed available record sets and fields, referencing entities by their `@id`.
- Loaded tabular data into DataFrames for analysis using record set `@id`.
- Performed EDA: filtered and normalized a numeric variable, grouped by a categorical key.
- Visualized the distribution and group comparisons of selected fields.

This structured approach ensures reproducibility and traceable provenance for dataset exploration with FAIR and Croissant standards.